# RAPID impact-classifier investigation

This notebook trains and inspects the discrete impact classifier. Labels are derived from the RAPID response only for training; geometry/resistance features are the model inputs.

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import subprocess
import sys
import pandas as pd
import matplotlib.pyplot as plt

REPO_ROOT = Path.cwd().resolve()
if not (REPO_ROOT / 'RAPID').exists():
    REPO_ROOT = REPO_ROOT.parents[1]
OUTPUT_ROOT = Path('/Volumes/PhD/river_hierarchy/network_variants/output')
EXAMPLE_IDS = ['02', '03', '05', '11', '21']  # edit when more real-network outputs are available
RUN_DIR = REPO_ROOT / 'RAPID' / 'classification' / 'results' / f"classifier-notebook-{datetime.now(timezone.utc):%Y%m%dT%H%M%SZ}"


In [ ]:
command = [sys.executable, str(REPO_ROOT / 'RAPID/classification/train_rapid_impact_classifier.py'),
           '--output-root', str(OUTPUT_ROOT), '--results-dir', str(RUN_DIR),
           '--example-ids', *EXAMPLE_IDS]
subprocess.run(command, cwd=REPO_ROOT, check=True)


In [ ]:
scores = pd.read_csv(RUN_DIR / 'grouped_cv_scores.csv')
confusion = pd.read_csv(RUN_DIR / 'grouped_cv_confusion.csv')
display(scores)


In [ ]:
recommended = scores.iloc[0]['model']
matrix = confusion.loc[confusion.model.eq(recommended)].pivot(index='actual', columns='predicted', values='count')
fig, ax = plt.subplots(figsize=(5, 4))
image = ax.imshow(matrix, cmap='Blues')
ax.set(xticks=range(len(matrix.columns)), xticklabels=matrix.columns, yticks=range(len(matrix.index)), yticklabels=matrix.index, xlabel='Predicted class', ylabel='Actual class', title=f'{recommended}: grouped-CV confusion matrix')
for i in range(matrix.shape[0]):
    for j in range(matrix.shape[1]):
        ax.text(j, i, int(matrix.iloc[i, j]), ha='center', va='center')
fig.colorbar(image, ax=ax, label='states')
plt.xticks(rotation=25, ha='right')
plt.show()
print((RUN_DIR / 'REPORT.md').read_text())
